# DSC 670 — Week 6 Exercise  
## Adapting the Chapter 9 Azure OpenAI Fine-Tuning Example to OpenAI

**Student:** Adjovi Lembila Akpaki  
**Course:** DSC670-T301 Advanced Uses of Generative AI  
**Assignment:** 6.2 Week 6 Exercise  

### Purpose

This notebook adapts the fine-tuning workflow presented in Chapter 9 of *Generative AI in Action* from Azure OpenAI to the standard OpenAI API. The workflow uses the supplied `Listing-9.1-emoji_ft_train.jsonl` dataset to fine-tune a chat model so that it returns text-based emoji labels such as `(devil)` instead of graphical emoji characters.

The exercise has five primary goals:

1. Inspect and validate the supplied JSONL dataset.
2. Upload the training data to OpenAI.
3. create a supervised fine-tuning job.
4. Monitor the remote training process through the OpenAI REST endpoint using Python's `requests` package.
5. Retrieve and test the resulting fine-tuned model.

The training computation occurs on OpenAI's servers. The local notebook is responsible for data preparation, API communication, status monitoring, and evaluation—not for performing the model training itself.


## 1. Environment Setup

The notebook uses the official OpenAI Python package for file upload, fine-tuning job creation, and model testing. The `requests` package is used separately to retrieve the fine-tuning job through its REST endpoint, which is a specific requirement of this assignment.

The installation command is kept inside the notebook so the work remains reproducible in a new Jupyter environment.


In [ ]:
%pip install --upgrade openai requests pandas

In [ ]:
import json
import os
import re
import time
from collections import Counter
from getpass import getpass
from pathlib import Path

import pandas as pd
import requests
from openai import OpenAI

print("Imports completed successfully.")

## 2. Secure API Authentication

The API key is entered with `getpass`, which prevents it from being displayed in the notebook output. Hard-coding a secret key in a notebook would create a security risk, especially when the file is submitted or shared.

Before submitting this notebook, no API key should appear in a code cell, output cell, screenshot, or filename.


In [ ]:
api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    api_key = getpass("Enter your OpenAI API key: ")

if not api_key or not api_key.startswith("sk-"):
    raise ValueError(
        "A valid OpenAI API key was not supplied. "
        "Set OPENAI_API_KEY or enter the key when prompted."
    )

client = OpenAI(api_key=api_key)

rest_headers = {
    "Authorization": f"Bearer {api_key}",
    "Content-Type": "application/json"
}

print("The OpenAI client is ready. The API key has not been displayed.")

## 3. Locate and Read the Supplied Dataset

The assignment explains that constructing a large dataset is not the main learning objective. Therefore, this notebook uses the supplied Chapter 9 file rather than inventing a replacement dataset.

Each nonblank line in a JSONL file must be an independent valid JSON object. Reading the file one line at a time makes it possible to identify the exact line responsible for a formatting error.


In [ ]:
training_file_path = Path("Listing-9.1-emoji_ft_train.jsonl")

if not training_file_path.exists():
    raise FileNotFoundError(
        f"{training_file_path.name} was not found. Place the dataset in "
        "the same folder as this notebook before continuing."
    )

records = []

with training_file_path.open("r", encoding="utf-8") as file:
    for line_number, line in enumerate(file, start=1):
        if not line.strip():
            continue

        try:
            records.append(json.loads(line))
        except json.JSONDecodeError as error:
            raise ValueError(
                f"Invalid JSON on line {line_number}: {error}"
            ) from error

print(f"Training file: {training_file_path.name}")
print(f"File size: {training_file_path.stat().st_size:,} bytes")
print(f"Number of nonblank JSONL records: {len(records)}")

## 4. Validate the Fine-Tuning Format

Successful parsing alone does not prove that the dataset is suitable for chat-model fine-tuning. Every example should contain a `messages` list with system, user, and assistant messages. Message roles and contents must also be valid.

This validation step reduces the chance of spending time and API credits on a job that fails because of an avoidable structural error.


In [ ]:
allowed_roles = {"system", "user", "assistant"}
required_roles = {"system", "user", "assistant"}
validation_errors = []

for example_number, record in enumerate(records, start=1):
    messages = record.get("messages")

    if not isinstance(messages, list) or not messages:
        validation_errors.append(
            f"Example {example_number}: 'messages' must be a nonempty list."
        )
        continue

    roles_in_example = set()

    for message_number, message in enumerate(messages, start=1):
        if not isinstance(message, dict):
            validation_errors.append(
                f"Example {example_number}, message {message_number}: "
                "message must be an object."
            )
            continue

        role = message.get("role")
        content = message.get("content")

        if role not in allowed_roles:
            validation_errors.append(
                f"Example {example_number}, message {message_number}: "
                f"invalid role {role!r}."
            )
        else:
            roles_in_example.add(role)

        if not isinstance(content, str) or not content.strip():
            validation_errors.append(
                f"Example {example_number}, message {message_number}: "
                "content must be a nonempty string."
            )

    missing_roles = required_roles - roles_in_example
    if missing_roles:
        validation_errors.append(
            f"Example {example_number}: missing roles {sorted(missing_roles)}."
        )

if validation_errors:
    print(f"Validation found {len(validation_errors)} problem(s).")
    for problem in validation_errors[:20]:
        print("-", problem)
    raise ValueError("The dataset did not pass validation.")

print("The dataset passed all structural validation checks.")

## 5. Exploratory Review of the Training Data

Fine-tuning quality depends on the examples used to teach the target behavior. Before uploading the data, I examined the size of the dataset, the consistency of the system instruction, the number of distinct target labels, and the distribution of those labels.

This review is not intended to prove model quality before training. Instead, it helps identify characteristics that may influence the final behavior, such as class imbalance, inconsistent formatting, or a large number of labels with only a few examples.


In [ ]:
system_prompts = Counter()
assistant_labels = Counter()
user_word_counts = []

for record in records:
    for message in record["messages"]:
        if message["role"] == "system":
            system_prompts[message["content"]] += 1
        elif message["role"] == "user":
            user_word_counts.append(len(message["content"].split()))
        elif message["role"] == "assistant":
            assistant_labels[message["content"]] += 1

dataset_summary = pd.DataFrame(
    {
        "Measure": [
            "Training examples",
            "Distinct system prompts",
            "Distinct assistant labels",
            "Average user-prompt length (words)",
            "Minimum user-prompt length (words)",
            "Maximum user-prompt length (words)"
        ],
        "Value": [
            len(records),
            len(system_prompts),
            len(assistant_labels),
            round(sum(user_word_counts) / len(user_word_counts), 2),
            min(user_word_counts),
            max(user_word_counts)
        ]
    }
)

dataset_summary

In [ ]:
label_distribution = pd.DataFrame(
    assistant_labels.most_common(),
    columns=["Assistant label", "Training examples"]
)

print("System prompt and frequency:")
for prompt, frequency in system_prompts.items():
    print(f"{frequency}: {prompt}")

print("\nTwenty most frequent target labels:")
display(label_distribution.head(20))

print("\nSample training record:")
print(json.dumps(records[0], indent=2, ensure_ascii=False))

### Dataset Commentary

The supplied file contains **569 examples**, and every record uses the same system instruction: *"You're a chatbot that only responds with emojis!"* This consistency is useful because it gives the model a stable behavioral objective.

The assistant outputs include **349 distinct text-based labels**. This is a relatively large output vocabulary compared with the number of examples, so many labels occur only a small number of times. That characteristic may limit generalization for rare labels. However, the assignment's purpose is to demonstrate adaptation and the fine-tuning workflow rather than to build a production-grade classifier.

The user prompts are short, which matches the intended conversational use case. The target outputs are also concise and consistently enclosed in parentheses. Consequently, formatting consistency is an important evaluation criterion in addition to whether the selected label is semantically reasonable.


## 6. Upload the JSONL File to OpenAI

The file is uploaded with the purpose `fine-tune`. OpenAI returns a file identifier, which is used when creating the training job. The identifier is not an API secret, but it is still stored in a variable so later cells can use it without manual copying.

The upload does not perform local training. It transfers the validated examples to the OpenAI platform so the remote fine-tuning service can process them.


In [ ]:
with training_file_path.open("rb") as training_file:
    uploaded_file = client.files.create(
        file=training_file,
        purpose="fine-tune"
    )

training_file_id = uploaded_file.id

print("Uploaded filename:", uploaded_file.filename)
print("Training file ID:", training_file_id)
print("Purpose:", uploaded_file.purpose)

## 7. Select the Base Model

The textbook example referenced an older Azure OpenAI workflow. OpenAI model availability changes over time, so the model name is isolated in one clearly labeled variable. The default below is a commonly used supervised fine-tuning model, but it should be changed if the OpenAI account or current documentation lists a different eligible model.

Keeping this selection explicit avoids silently using an outdated model name from the book.


In [ ]:
# Change this value only if your OpenAI account or the current official
# fine-tuning documentation requires a different supported base model.
base_model = "gpt-4o-mini-2024-07-18"

print("Selected base model:", base_model)

## 8. Create the Fine-Tuning Job

The training file identifier and base-model name are submitted to OpenAI to create a supervised fine-tuning job. I allow OpenAI to choose the default hyperparameters because the objective is to reproduce and adapt the instructional example rather than conduct a broad hyperparameter experiment.

The returned job identifier is essential. It is used in the REST URL required by the assignment and later provides access to the final fine-tuned model name.


In [ ]:
fine_tuning_job = client.fine_tuning.jobs.create(
    training_file=training_file_id,
    model=base_model,
    suffix="dsc670-emoji"
)

job_id = fine_tuning_job.id

print("Fine-tuning job ID:", job_id)
print("Initial job status:", fine_tuning_job.status)
print("Base model:", fine_tuning_job.model)

## 9. Monitor Training Through the REST Endpoint

This section directly fulfills the assignment requirement to use the Python `requests` package to check the model-training process.

The REST endpoint follows this pattern:

`https://api.openai.com/v1/fine_tuning/jobs/{job_id}`

The loop retrieves the job, reports changes in status, and waits between calls. Polling every 60 seconds is frequent enough to observe progress without sending unnecessary requests. The loop stops only when the job reaches a terminal state: `succeeded`, `failed`, or `cancelled`.

Because training may take a considerable amount of time, the cell can be interrupted and rerun later as long as `job_id` is still available. If the notebook kernel is restarted, the job ID can be manually restored in the recovery cell below.


In [ ]:
# Recovery option:
# If the kernel was restarted after job creation, uncomment the next line
# and paste the previously printed job ID.
# job_id = "ftjob-REPLACE_WITH_YOUR_JOB_ID"

job_status_url = (
    f"https://api.openai.com/v1/fine_tuning/jobs/{job_id}"
)

terminal_statuses = {"succeeded", "failed", "cancelled"}
previous_status = None

while True:
    try:
        response = requests.get(
            job_status_url,
            headers=rest_headers,
            timeout=30
        )
        response.raise_for_status()
    except requests.RequestException as error:
        raise RuntimeError(
            f"Unable to retrieve the fine-tuning job: {error}"
        ) from error

    job_information = response.json()
    current_status = job_information.get("status", "unknown")

    if current_status != previous_status:
        timestamp = time.strftime("%Y-%m-%d %H:%M:%S")
        print(f"{timestamp} | Status: {current_status}")
        previous_status = current_status

    if current_status in terminal_statuses:
        break

    time.sleep(60)

print("\nThe monitoring loop has ended.")
print("Final status:", current_status)

## 10. Inspect the Final REST Response and Retrieve the Model Name

A successful job response includes the `fine_tuned_model` field. This value is the exact model identifier required for inference. Retrieving it programmatically is safer than manually copying a name from a dashboard because it preserves the connection between the monitored job and the tested model.

If the job fails, the notebook displays the available error information rather than pretending that training completed successfully.


In [ ]:
print(json.dumps(job_information, indent=2))

if current_status != "succeeded":
    error_information = job_information.get("error")
    raise RuntimeError(
        "The fine-tuning job did not succeed. "
        f"Final status: {current_status}. Error: {error_information}"
    )

fine_tuned_model = job_information.get("fine_tuned_model")

if not fine_tuned_model:
    raise RuntimeError(
        "The job succeeded, but the response did not include a "
        "'fine_tuned_model' value."
    )

print("\nFine-tuned model name:")
print(fine_tuned_model)

## 11. Test the Fine-Tuned Model With the Instructor-Provided Pattern

The following test follows the code pattern supplied with the assignment. The placeholder model name is replaced with the value returned by the completed fine-tuning job.

The prompt, *"What the hell is going on?"*, tests whether the model applies its learned text-label convention to wording that expresses confusion or frustration. The important question is not merely whether the answer relates to an emoji. The answer should also follow the specialized parenthesized format represented in the training data.


In [ ]:
completion = client.chat.completions.create(
    model=fine_tuned_model,
    messages=[
        {
            "role": "system",
            "content": "You're a chatbot that only responds with emojis!"
        },
        {
            "role": "user",
            "content": "What the hell is going on?"
        }
    ],
    max_tokens=50,
    temperature=0
)

instructor_test_output = completion.choices[0].message.content

print("Fine-tuned model output:")
print(instructor_test_output)

## 12. Compare the Base and Fine-Tuned Models

A single successful answer is not enough to evaluate adaptation. This comparison sends the same unseen prompts to both the base model and the fine-tuned model. Using identical instructions and a temperature of zero makes the comparison more controlled and repeatable.

The evaluation considers three questions:

1. Does the response use the expected parenthesized label format?
2. Does it avoid unnecessary explanation?
3. Is the chosen label semantically plausible for the prompt?

The prompts below represent different emotions and situations and are not copied directly from the displayed first training record.


In [ ]:
test_prompts = [
    "What the hell is going on?",
    "I cannot believe I finally passed the exam!",
    "That joke was absolutely hilarious.",
    "I feel nervous about tomorrow's interview.",
    "Please do not tell anyone my secret.",
    "That was a very evil thing to do.",
    "I am exhausted and ready for bed.",
    "The situation is confusing me."
]

def obtain_model_response(model_name, user_prompt):
    response = client.chat.completions.create(
        model=model_name,
        messages=[
            {
                "role": "system",
                "content": "You're a chatbot that only responds with emojis!"
            },
            {
                "role": "user",
                "content": user_prompt
            }
        ],
        max_tokens=50,
        temperature=0
    )
    return response.choices[0].message.content.strip()

comparison_rows = []

for prompt in test_prompts:
    base_output = obtain_model_response(base_model, prompt)
    fine_tuned_output = obtain_model_response(fine_tuned_model, prompt)

    comparison_rows.append(
        {
            "Prompt": prompt,
            "Base model output": base_output,
            "Fine-tuned model output": fine_tuned_output
        }
    )

results_df = pd.DataFrame(comparison_rows)
results_df

## 13. Evaluate Formatting Consistency

The training targets generally follow a compact pattern such as `(party)` or `(devil)`. The following simple check does not determine whether a label is emotionally correct, but it provides an objective measure of whether each model follows the expected output convention.

This distinction matters because the base model may understand the prompt while still returning a graphical emoji, a sentence, or several alternatives. The adaptation is successful only when it changes the model's response behavior in the intended direction.


In [ ]:
label_pattern = re.compile(r"^\([A-Za-z0-9_\- ]+\)$")

results_df["Base format match"] = results_df[
    "Base model output"
].apply(lambda value: bool(label_pattern.fullmatch(value)))

results_df["Fine-tuned format match"] = results_df[
    "Fine-tuned model output"
].apply(lambda value: bool(label_pattern.fullmatch(value)))

format_summary = pd.DataFrame(
    {
        "Model": ["Base model", "Fine-tuned model"],
        "Correctly formatted outputs": [
            int(results_df["Base format match"].sum()),
            int(results_df["Fine-tuned format match"].sum())
        ],
        "Total test prompts": [len(results_df), len(results_df)]
    }
)

format_summary["Format compliance rate"] = (
    format_summary["Correctly formatted outputs"]
    / format_summary["Total test prompts"]
)

display(results_df)
display(format_summary)

## 14. Results Commentary — Complete After Running the API Cells

Replace the bracketed items below with the values and examples shown in the output tables. This section must describe the **actual results obtained**; it should not claim that training succeeded before the notebook has been run.

The fine-tuning job finished with the status **[insert final status]**. The REST response returned the model identifier **[insert fine-tuned model name]**, confirming that the job produced a model that could be called through the Chat Completions API.

For the instructor-provided prompt, *"What the hell is going on?"*, the fine-tuned model returned **[insert exact response]**. This output **[did/did not]** follow the parenthesized text-label convention represented in the training examples.

Across the eight comparison prompts, the base model followed the required format in **[insert number] out of 8** cases, while the fine-tuned model followed it in **[insert number] out of 8** cases. The strongest evidence of adaptation was **[describe one specific comparison from the table]**. In that example, the base model produced **[insert base output]**, whereas the fine-tuned model produced **[insert fine-tuned output]**.

The semantic quality of the labels was **[consistent/mixed/inconsistent]**. For example, **[discuss one strong result]**. A weaker or ambiguous result occurred for **[identify prompt]**, where the model returned **[output]**. That result may reflect the dataset's large number of distinct labels and the small number of examples available for many labels.

Overall, the experiment indicates that fine-tuning **[did/did not]** improve adherence to the specialized response format. The comparison also shows why model evaluation should consider both formatting and meaning. A correctly parenthesized answer is not necessarily a good answer if the selected label does not fit the prompt.


## 15. Limitations and Opportunities for Improvement

The dataset is appropriate for demonstrating the OpenAI fine-tuning process, but it has limitations. It contains 349 distinct assistant labels across 569 examples, which means many labels have only limited representation. A production system would benefit from additional examples for rare labels, more balanced label frequencies, and a separate validation dataset.

The short prompts also do not fully represent difficult language phenomena such as sarcasm, mixed emotions, indirect expressions, or culturally dependent interpretations. Future work could test those cases systematically and compare multiple fine-tuning settings.

A stronger evaluation could include:

- A held-out labeled test set not used during training.
- Exact-match accuracy against expected labels.
- Separate measurements for format compliance and semantic correctness.
- Error analysis grouped by frequent and rare labels.
- Repeated tests to assess output stability.

These additions would make it easier to distinguish genuine generalization from memorization or formatting imitation.


## 16. Conclusion

This notebook adapted the Chapter 9 Azure OpenAI example to the standard OpenAI API. The workflow validated and explored the supplied JSONL dataset, uploaded it for fine-tuning, created a supervised training job, monitored that job through a REST endpoint with `requests`, retrieved the resulting model name, and tested the customized model.

The central purpose of the exercise was not to prove that a general language model can discuss emotions or produce graphical emojis. The base model already has those abilities. The meaningful adaptation was teaching the model to respond consistently using a specialized text-based convention such as `(devil)`.

The base-versus-fine-tuned comparison provides evidence about whether the training changed that behavior. At the same time, the dataset review and limitations discussion show that successful API execution alone is not sufficient evidence of model quality. A responsible evaluation must examine consistency, relevance, generalization, and the characteristics of the training data.


## References

Bahree, A. (2024). *Generative AI in action*. Manning Publications.

OpenAI. (n.d.). *Fine-tuning API reference*. OpenAI Platform documentation. https://platform.openai.com/docs/api-reference/fine-tuning

OpenAI. (n.d.). *Files API reference*. OpenAI Platform documentation. https://platform.openai.com/docs/api-reference/files
